```
cd /Users/noahwebb/Python/scraping_env2
source bin/activate



python3 -m streamlit run pokeScrape/app.py
```


# Process Overview

```mermaid


graph LR
    A[<small>Start</small>] --> B[<small>Scrape Website<br/>Data</small>]
    B --> C[<small>Store in<br/>Database</small>]
    C --> D[<small>Read from<br/>Database</small>]
    D --> E[<small>Export to<br/>CSV</small>]
    E --> F[<small>Run App<br/>with Table</small>]
    F --> G[<small>Publish App<br/>to Website</small>]
    G --> H[<small>End</small>]




In [ ]:
import requests
from bs4 import BeautifulSoup 
import pandas as pd 
import re
import os




: 

# Scraping serebii.net

In [ ]:
def getPage(url):
    """
    Utilty function used to get a Beautiful Soup object from a given URL
    """

    session = requests.Session()
    headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_9_5) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/39.0.2171.95 Safari/537.36',
               'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8'}
    try:
        req = session.get(url, headers=headers)
    except requests.exceptions.RequestException:
        return None
    bs = BeautifulSoup(req.text, 'html.parser')
    return bs

In [ ]:

# page = 'https://fantasy.espn.com/football/leaders'
page = 'https://www.serebii.net/pokemon/nationalpokedex.shtml'

bs = getPage(page)

print(bs.prettify())

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = 'https://www.serebii.net/pokemon/nationalpokedex.shtml'
response = requests.get(url)
bs = BeautifulSoup(response.content, "html.parser")

table = bs.find("table")

# Store table data here
table_data = []

# Skip the first two rows, then remove odd-numbered ones
rows = table.find_all("tr")[2:]  # Remove first 2 header rows
rows = rows[::2]  # Keep only even-indexed rows (0, 2, 4, ...)


for row in rows:
    cells = row.find_all(["th", "td"])
    cell_data = []

    for cell in cells:
        links = cell.find_all("a")
        if links:
            link_entries = [f"{a.get_text(strip=True)} ({a['href']})" for a in links if a.has_attr("href")]
            cell_data.append(" | ".join(link_entries))
        else:
            cell_data.append(cell.get_text(strip=True))

    if cell_data:  # Avoid appending empty rows
        table_data.append(cell_data)

# Create DataFrame
df = pd.DataFrame(table_data)

# Optionally set headers if you've created them
# df.columns = ['No. - Pic', 'Name', 'Type', 'Abilities', 'Base Stats', 'HP - Att', 'Def', 'S.Att', 'S.Def', 'Spd']

df.head(10)


In [ ]:
row1 = ['No. - Pic', 'Name', 'Type', 'Abilities', 'Base Stats']
row2 = ['HP - Att', 'Def', 'S.Att', 'S.Def', 'Spd']

# Simply concatenate the lists
combined_row = row1 + row2

print(combined_row)


In [ ]:
manual_header = ['No.', 'Pic', 'remove','Name', 'Type', 'Abilities', 'HP', 'Att', 'Def', 'S.Att', 'S.Def', 'Spd']
df.columns = manual_header
df.head(10)

In [ ]:
stat_cols = ["HP", "Att", "Def", "S.Att", "S.Def", "Spd"]
df["Total"] = df[stat_cols].astype(int).sum(axis=1)
df.head(10)

# Clean Data

In [ ]:
# df_sorted = df.sort_values(by="Total", ascending=False)
df.drop(columns=["remove", "Pic"], inplace=True)


In [ ]:
def extract_type_basenames(type_column):
    def get_basenames(s):
        paths = re.findall(r'\(([^)]+)\)', s)
        return ' | '.join([os.path.basename(path) for path in paths])
    return type_column.apply(get_basenames)

# Example usage:
df['Type'] = extract_type_basenames(df['Type'])

# Loading to MySQL database

## Connecting to database/server

In [ ]:
# ...existing code...
from sqlalchemy import create_engine

# Replace with your actual MySQL credentials and database name
user = 'testuser'
password = 'test1963$'
host = '127.0.0.1'  # or your MySQL server address
# host = 'Noahs-MacBook-Air.local'
# database = 'test'
database = 'pokewebb_test'

# Create SQLAlchemy engine
engine = create_engine(f'mysql+pymysql://{user}:{password}@{host}/{database}')

print(f"connected to {database} database as {user} on {host}.")


# NOTE
# User did not have priveledges to connect to 

# The error (1044, "Access denied for user 'testuser'@'localhost' to database 'test'") 
# means your MySQL user testuser does not have permission to access the test database.

# To fix this, you need to grant the necessary privileges in MYSQL to the user by the following code: 

# GRANT ALL PRIVILEGES ON test.* TO 'testuser'@'localhost';
# FLUSH PRIVILEGES;


## Writing to database table

In [ ]:
# Write DataFrame to MySQL table (replace 'pokemon_table' with your table name)
df.to_sql('serbii', con=engine, if_exists='replace', index=False)
# if_exists='replace' will overwrite the table; use 'append' to add data instead

print("DataFrame loaded to MySQL table successfully.")
# ...existing code...

## Reading database table

In [ ]:
import pandas as pd
from tabulate import tabulate

# Run query and load into DataFrame
df = pd.read_sql("""
                 SELECT * 
                 FROM serbii 
                 """
                 , engine)

print(tabulate(df, headers='keys', tablefmt='pretty', showindex=False))

# Exporting df to a CSV

In [ ]:
from datetime import datetime
import os 

# Define filename and file extension
current_time = datetime.now().strftime("%Y%m%d_%H%M%S")
folder = 'serbii_data'
path = '/Users/noahwebb/Python/scraping_env2/pokeScrape/'
base_file_name = f'serbii_data'
file_extension = '.csv'

# create folder if it does not exist
folder_path = os.path.join(path, folder)
os.makedirs(folder_path, exist_ok=True)

# Create full file path
file_name = f"{base_file_name}_{current_time}{file_extension}"  # Create a unique filename with timestamp


file_path = os.path.join(folder_path, file_name)

print(f"Saving DataFrame to {file_path}")
df.to_csv(file_path, index=False)
print("DataFrame saved to CSV file successfully.")

# Read CSV file

In [ ]:
import re 

# Directory containing the files
folder = "/Users/noahwebb/Python/scraping_env2/pokeScrape/serbii_data"


# List all files in the folder
all_files = os.listdir(folder)
# print(all_files)

# Function to extract date from filename
def extract_date(filename):
    match = re.match(r"serbii_data_(\d{8})_(\d{6})\.csv", filename)
    return int(match.group(1)) if match else None

# Collect valid files with extracted dates
file_data = []
for f in all_files:
    date = extract_date(f)
    if date:
        file_data.append({'file': os.path.join(folder, f), 'date': date})

# Convert to DataFrame and sort
df_files = pd.DataFrame(file_data)
df_files_sorted = df_files.sort_values(by='date', ascending=False)
print(df_files_sorted.head(10))


In [ ]:
# Load the latest file
if not df_files_sorted.empty:
    latest_file = df_files_sorted.iloc[0]['file']
    df = pd.read_csv(latest_file)
    print(f"Loaded: {latest_file}")
else:
    raise FileNotFoundError("No matching files found.")

csv_df = pd.read_csv(latest_file)  # Or build from scraped data


In [ ]:
print(tabulate(csv_df, headers='keys', tablefmt='pretty', showindex=False))